In [86]:
import pandas as pd
import geopandas as gpd
from shapely import Point, MultiPoint
from sklearn.cluster import DBSCAN

In [87]:
# Load birds data into a pandas dataframe
input_csv = pd.read_csv('../data/black-tailed.csv', index_col=0)

# Get list of birds names
NAMES = sorted(list(set(input_csv['individual-local-identifier'])))

# Set a filter to select only the birds in Rotterdam
input_csv = input_csv[input_csv['individual-local-identifier'] == 'Rotterdam']

# Remove rows that have no coordinates
data = input_csv[input_csv['location-long'].notnull()]

In [88]:
# Create a geometry column
birds_locations = data.apply(lambda x: Point(x['location-long'], x['location-lat']), axis=1)

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry=birds_locations)
gdf = gdf.set_crs('epsg:4326')

# Project to Web Mercator
gdf = gdf.to_crs('epsg:3857')

In [89]:
COORDS = []
for i in range(len(gdf)):
    c = [list(gdf.geometry.x)[i], list(gdf.geometry.y)[i]]
    COORDS.append(c)

In [90]:
# Compute DBSCAN
clustering = DBSCAN(eps=15000, min_samples=25).fit(COORDS)
labels = clustering.labels_

In [91]:
# Create a new table
clusterLayer = gpd.GeoDataFrame(columns=['cluster_id', 'number_of_points', 'geometry'])

for cluster_id in set(labels):
    if cluster_id == -1:
        continue
    POINTS = []
    for i in range(len(labels)):
        if labels[i] == cluster_id:
            coord = COORDS[i]
            point = Point(coord[0], coord[1])
            POINTS.append(point)
            geomCluster = MultiPoint(POINTS)
            clusterLayer = gpd.GeoDataFrame(pd.concat([clusterLayer, gpd.GeoDataFrame({'cluster_id': cluster_id, 'number_of_points': len(POINTS), 'geometry': geomCluster}, index=[0])], ignore_index=True))

clusterLayer = clusterLayer.set_crs('epsg:3857')